In [ ]:
"""
chute_drop_batch.py

Batch Monte-Carlo chute-drop validation harness.

Reads the geometric pose analysis produced by ChutePoseAnalysis.m (MATLAB)
for one or more parts -- specifically each condition's
"<folder>_MASTER_summary.txt" -- and, for every (part, roll, pitch)
combination requested, runs N randomized PyBullet drop trials into a chute
tilted to that same roll/pitch. Each trial's settled orientation is matched
(in the chute's LEVEL/untilted local frame, exactly as ChutePoseAnalysis.m
assumes) against the refQuat catalog for that condition, and match counts
are tallied into an empirical pose-frequency distribution that can be
compared directly against the CSA_B / CSA_A / CSA_N / CRSA_B / CRSA_A /
CRSA_N columns already computed by the MATLAB analysis.

--------------------------------------------------------------------------
EXPECTED DIRECTORY LAYOUT (as produced by ChutePoseAnalysis.m)
--------------------------------------------------------------------------
    <parts_folder>/
        PartA.stl
        PartB.stl
        results/
            PartA/
                PartA_15R_15P/
                    PartA_15R_15P_MASTER_summary.txt
                    PartA_15R_15P_GEOMETRIC_summary.txt
                    PartA_15R_15P.pdf
                    PartA_15R_15P_poseCatalog.csv      <- OPTIONAL, see below
                PartA_15R_25P/
                    ...
            PartB/
                ...

--------------------------------------------------------------------------
POSE CATALOG (symmetric-variant matching)
--------------------------------------------------------------------------
_MASTER_summary.txt stores exactly ONE representative refQuat per merged
stable pose -- it does NOT retain the other symmetric-orientation variants
that were folded into that pose during mergeByCSAValues (only their count,
"nMerge"). Matching a dropped part's final orientation against a single
representative quaternion will under-match any pose that has rotational
symmetry (e.g. a square face has 4 equivalent resting orientations 90 deg
apart in quaternion space -- far outside quatMatchTol).

For correct matching this script looks for a companion CSV next to the
MASTER_summary:  "<folder>_poseCatalog.csv"  (or anything matching
"*catalog*.csv" in that folder), with columns pose_id,qw,qx,qy,qz and one
row per symmetric variant (this is the "exportPoseQuatCatalog.m" catalog
referenced in the original chute_drop_sim.py). If found, ALL variants are
checked when matching, same as MATLAB's p4_matchQuatComposed.

If no catalog CSV is found, this script falls back to matching against the
single MASTER_summary refQuat per pose and prints a warning -- results
will systematically under-count symmetric poses in that case.

--------------------------------------------------------------------------
ANGLE GRID
--------------------------------------------------------------------------
    alphas (roll)  = 15, 17.5, 20   deg
    betas  (pitch) = 15, 25, 35, 45 deg

The chute is physically tilted using the EXACT alpha/beta values above.
The results-folder name is looked up using MATLAB's round() convention
(round-half-away-from-zero), matching ChutePoseAnalysis.m's
`sprintf('%s_%dR_%dP', partName, round(chuteRoll_deg), round(chutePitch_deg))`
-- so alpha=17.5 physically tilts the chute 17.5 deg but looks up folder
tag "18R". This mirrors the MATLAB code's own math/display discrepancy.

--------------------------------------------------------------------------
CONTACT MODEL -- PTFE chute (perforated, textured) / PTFE-coated part
--------------------------------------------------------------------------
Both bodies are nominally near-frictionless PTFE-on-PTFE (mu ~ 0.04-0.15),
but the chute is perforated and surface-textured, so the *effective*
contact friction it presents is not a single constant -- hole rims and
rough patches can grip noticeably more than the bulk PTFE surface, and a
part edge can briefly catch on a hole. Rather than modelling hole geometry
explicitly, this script:
  - redraws chute and part friction/restitution from configurable bands
    EVERY trial (--chute-friction-*, --part-friction-*, --*-restitution-*)
  - with probability --snag-prob, additionally boosts that trial's chute
    friction by --snag-friction-boost, representing an edge catching on a
    hole rim/burr
  - sets low rolling/spinning friction for both bodies (PTFE has very low
    rolling resistance), via --rolling-friction / --spinning-friction

This is a coarse statistical stand-in for hole/texture geometry, not a
geometric model of the perforations -- flagged here explicitly since it's
a simplification.

--------------------------------------------------------------------------
STABILITY THRESHOLD CROSS-REFERENCE
--------------------------------------------------------------------------
Every condition's THRESH_WALL / THRESH_FLOOR (parsed from the
MASTER_summary header) is carried into the output, along with each
matched pose's ratioWall/ratioFloor and MATLAB's own
TRANSITIONS / FLOOR-UNSTABLE / Q=0 notes, so simulated hits on a pose the
geometric analysis flagged as invalid/unstable are visible in the summary
rather than silently folded into the count.

--------------------------------------------------------------------------
USAGE
--------------------------------------------------------------------------
    python chute_drop_batch.py \
        --parts-folder /path/to/partsFolder \
        --chute chute.obj --chute-concave \
        --n 1000 \
        --out-dir batch_drop_results

    # just see what conditions would run, without simulating:
    python chute_drop_batch.py --parts-folder ... --chute chute.obj --dry-run
"""

import argparse
import csv
import glob
import os
import re
import time
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple

import numpy as np
import pybullet as p
import pybullet_data


# ===========================================================================
# Quaternion primitives (kept in [w,x,y,z] order to match MATLAB exactly,
# converted to PyBullet's [x,y,z,w] only at the pybullet API boundary)
# ===========================================================================

def q_from_axis_angle_wxyz(axis, angle):
    axis = np.asarray(axis, dtype=float)
    axis = axis / np.linalg.norm(axis)
    s = np.sin(angle / 2.0)
    return np.array([np.cos(angle / 2.0), s * axis[0], s * axis[1], s * axis[2]])


def q_compose_wxyz(q1, q2):
    """Matches ChutePoseAnalysis.m's q_compose(q1, q2) exactly."""
    w1, x1, y1, z1 = q1
    w2, x2, y2, z2 = q2
    return np.array([
        w2 * w1 - x2 * x1 - y2 * y1 - z2 * z1,
        w2 * x1 + x2 * w1 + y2 * z1 - z2 * y1,
        w2 * y1 - x2 * z1 + y2 * w1 + z2 * x1,
        w2 * z1 + x2 * y1 - y2 * x1 + z2 * w1,
    ])


def wxyz_to_xyzw(q_wxyz):
    return (float(q_wxyz[1]), float(q_wxyz[2]), float(q_wxyz[3]), float(q_wxyz[0]))


def q_geodesic_wxyz(q1_wxyz, q2_wxyz):
    """Sign-invariant geodesic distance, matches MATLAB's q_geodesic."""
    dp = abs(np.dot(q1_wxyz, q2_wxyz))
    dp = min(dp, 1.0)
    return 2.0 * np.arccos(dp)


def random_quaternion_xyzw(rng: np.random.Generator):
    """Uniformly random unit quaternion (Shoemake's method) -> (x,y,z,w)."""
    u1, u2, u3 = rng.random(3)
    q_w = np.sqrt(1 - u1) * np.sin(2 * np.pi * u2)
    q_x = np.sqrt(1 - u1) * np.cos(2 * np.pi * u2)
    q_y = np.sqrt(u1) * np.sin(2 * np.pi * u3)
    q_z = np.sqrt(u1) * np.cos(2 * np.pi * u3)
    return (q_x, q_y, q_z, q_w)


def build_chute_quat_xyzw(roll_deg: float, pitch_deg: float):
    """Reproduces ChutePoseAnalysis.m's chute frame exactly:
        roll  = -deg2rad(chuteRoll_deg)   about X
        pitch =  deg2rad(chutePitch_deg)  about Y
        q_chute = q_compose(q_roll, q_pitch)
    Returned in PyBullet (x,y,z,w) order.
    """
    roll = -np.deg2rad(roll_deg)
    pitch = np.deg2rad(pitch_deg)
    q_roll = q_from_axis_angle_wxyz([1, 0, 0], roll)
    q_pitch = q_from_axis_angle_wxyz([0, 1, 0], pitch)
    q_chute = q_compose_wxyz(q_roll, q_pitch)
    q_chute = q_chute / np.linalg.norm(q_chute)
    return wxyz_to_xyzw(q_chute)


def rotate_to_chute_local(final_quat_xyzw, chute_quat_xyzw):
    """Express a world-frame orientation in the chute's local (level) frame:
    q_local = q_chute^-1 * q_final. Required because refQuats in the
    MATLAB catalog assume a level chute (no Rchute tilt baked in).
    """
    q_chute_inv = p.invertTransform([0, 0, 0], chute_quat_xyzw)[1]
    _, q_local = p.multiplyTransforms([0, 0, 0], q_chute_inv, [0, 0, 0], final_quat_xyzw)
    return q_local


def rotate_vec_by_quat_xyzw(vec, quat_xyzw):
    pos, _ = p.multiplyTransforms([0, 0, 0], quat_xyzw, vec, [0, 0, 0, 1])
    return pos


def matlab_round(x: float) -> int:
    """Round-half-away-from-zero, matching MATLAB's round()."""
    return int(np.floor(abs(x) + 0.5) * (1 if x >= 0 else -1))


# ===========================================================================
# Pose catalog (symmetric variants) -- from exportPoseQuatCatalog.m CSV
# ===========================================================================

def load_pose_catalog(csv_path: str) -> Dict[int, np.ndarray]:
    """pose_id -> Nx4 array of quaternions in [w,x,y,z] order."""
    catalog: Dict[int, list] = {}
    with open(csv_path, newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            pose_id = int(row["pose_id"])
            q_wxyz = [float(row["qw"]), float(row["qx"]), float(row["qy"]), float(row["qz"])]
            catalog.setdefault(pose_id, []).append(q_wxyz)
    return {k: np.array(v) for k, v in catalog.items()}


def find_catalog_csv(condition_dir: str, folder_name: str) -> Optional[str]:
    candidates = [
        os.path.join(condition_dir, f"{folder_name}_poseCatalog.csv"),
        os.path.join(condition_dir, f"{folder_name}_pose_catalog.csv"),
    ]
    for c in candidates:
        if os.path.isfile(c):
            return c
    matches = sorted(glob.glob(os.path.join(condition_dir, "*catalog*.csv")))
    return matches[0] if matches else None


def match_pose_in_catalog(final_quat_xyzw, catalog: Dict[int, np.ndarray], tol: float):
    x, y, z, w = final_quat_xyzw
    q_wxyz = np.array([w, x, y, z])
    best_id, best_d = None, np.inf
    for pose_id, variants in catalog.items():
        for q_ref in variants:
            d = q_geodesic_wxyz(q_wxyz, q_ref)
            if d < best_d:
                best_d, best_id = d, pose_id
    if best_id is not None and best_d <= tol:
        return best_id, best_d
    return None, best_d


# ===========================================================================
# MASTER_summary.txt parser (metadata: CSA/CRSA weights, thresholds, notes)
# ===========================================================================

POSE_ROW_RE = re.compile(
    r'^\s*(?P<si>\d+)\s+(?P<plane>\d+)\s+(?P<theta>-?\d+)\s+'
    r'\[\s*(?P<qw>-?\d+\.\d+)\s+(?P<qx>-?\d+\.\d+)\s+(?P<qy>-?\d+\.\d+)\s+(?P<qz>-?\d+\.\d+)\]\s+'
    r'(?P<omega>-?\d+\.\d+)\s+(?P<height>-?\d+\.\d+)\s+'
    r'(?P<rwall>-?\d+\.\d+)\s+(?P<rfloor>-?\d+\.\d+)\s+'
    r'(?P<csa_b>-?\d+\.\d+)\s+(?P<csa_a>-?\d+\.\d+)\s+(?P<csa_n>-?\d+\.\d+)\s+'
    r'(?P<crsa_b>-?\d+\.\d+)\s+(?P<crsa_a>-?\d+\.\d+)\s+(?P<crsa_n>-?\d+\.\d+)\s+'
    r'(?P<dest>\S+)\s+(?P<from>\S+)\s+(?P<nmerge>\d+)\s+(?P<notes>.*?)\s*$'
)

THRESH_WALL_RE = re.compile(r'Wall\s+transition threshold\s*:\s*([\d.]+)')
THRESH_FLOOR_RE = re.compile(r'Floor instability threshold\s*:\s*([\d.]+)')
SUMQ_RE = re.compile(r'Sum of raw CSA weights.*?:\s*([\-\d.]+)')


@dataclass
class PoseEntry:
    si: int
    floor_plane: int
    theta_deg: float
    quat_wxyz: np.ndarray
    omega: float
    height: float
    ratio_wall: float
    ratio_floor: float
    csa_b: float
    csa_a: float
    csa_n: float
    crsa_b: float
    crsa_a: float
    crsa_n: float
    dest: Optional[int]
    received_from: Optional[int]
    n_merge: int
    notes: str

    @property
    def flag_transitions(self) -> bool:
        return "TRANSITIONS" in self.notes

    @property
    def flag_floor_unstable(self) -> bool:
        return "FLOOR-UNSTABLE" in self.notes

    @property
    def flag_zero_weight(self) -> bool:
        return self.notes.strip() not in ("", "-")


@dataclass
class ConditionData:
    part: str
    alpha: float          # actual physical roll used to tilt the chute
    beta: float            # actual physical pitch used to tilt the chute
    roll_tag: int           # MATLAB-rounded folder tag
    pitch_tag: int
    condition_dir: str
    folder_name: str
    thresh_wall: float
    thresh_floor: float
    sum_q: float
    poses: Dict[int, PoseEntry] = field(default_factory=dict)
    catalog: Optional[Dict[int, np.ndarray]] = None
    catalog_path: Optional[str] = None


def parse_master_summary(master_path: str) -> Tuple[Dict[int, PoseEntry], float, float, float]:
    poses: Dict[int, PoseEntry] = {}
    thresh_wall = 2.731   # ChutePoseAnalysis.m defaults, used if header parse fails
    thresh_floor = 2.296
    sum_q = 0.0

    with open(master_path, "r") as f:
        for line in f:
            m = THRESH_WALL_RE.search(line)
            if m:
                thresh_wall = float(m.group(1))
                continue
            m = THRESH_FLOOR_RE.search(line)
            if m:
                thresh_floor = float(m.group(1))
                continue
            m = SUMQ_RE.search(line)
            if m:
                sum_q = float(m.group(1))
                continue

            m = POSE_ROW_RE.match(line)
            if not m:
                continue
            g = m.groupdict()
            si = int(g["si"])
            dest = None if g["dest"] == "-" else int(g["dest"])
            frm = None if g["from"] == "-" else int(g["from"])
            poses[si] = PoseEntry(
                si=si,
                floor_plane=int(g["plane"]),
                theta_deg=float(g["theta"]),
                quat_wxyz=np.array([float(g["qw"]), float(g["qx"]), float(g["qy"]), float(g["qz"])]),
                omega=float(g["omega"]),
                height=float(g["height"]),
                ratio_wall=float(g["rwall"]),
                ratio_floor=float(g["rfloor"]),
                csa_b=float(g["csa_b"]), csa_a=float(g["csa_a"]), csa_n=float(g["csa_n"]),
                crsa_b=float(g["crsa_b"]), crsa_a=float(g["crsa_a"]), crsa_n=float(g["crsa_n"]),
                dest=dest, received_from=frm, n_merge=int(g["nmerge"]), notes=g["notes"].strip(),
            )

    return poses, thresh_wall, thresh_floor, sum_q


# ===========================================================================
# Condition discovery
# ===========================================================================

def discover_conditions(parts_folder: str, alphas: List[float], betas: List[float],
                         parts_filter: Optional[List[str]]) -> List[ConditionData]:
    results_root = os.path.join(parts_folder, "results")
    if not os.path.isdir(results_root):
        raise FileNotFoundError(f"No 'results' folder found under {parts_folder}")

    part_dirs = sorted(
        d for d in os.listdir(results_root)
        if os.path.isdir(os.path.join(results_root, d))
    )
    if parts_filter:
        part_dirs = [d for d in part_dirs if d in parts_filter]

    conditions: List[ConditionData] = []
    for part in part_dirs:
        for alpha in alphas:
            for beta in betas:
                roll_tag = matlab_round(alpha)
                pitch_tag = matlab_round(beta)
                folder_name = f"{part}_{roll_tag}R_{pitch_tag}P"
                condition_dir = os.path.join(results_root, part, folder_name)
                master_path = os.path.join(condition_dir, f"{folder_name}_MASTER_summary.txt")

                if not os.path.isfile(master_path):
                    print(f"  [skip] no MASTER_summary for {part} alpha={alpha} beta={beta} "
                          f"(expected {master_path})")
                    continue

                poses, thresh_wall, thresh_floor, sum_q = parse_master_summary(master_path)
                if not poses:
                    print(f"  [skip] {master_path} parsed but contained no stable poses")
                    continue

                cond = ConditionData(
                    part=part, alpha=alpha, beta=beta,
                    roll_tag=roll_tag, pitch_tag=pitch_tag,
                    condition_dir=condition_dir, folder_name=folder_name,
                    thresh_wall=thresh_wall, thresh_floor=thresh_floor, sum_q=sum_q,
                    poses=poses,
                )

                catalog_path = find_catalog_csv(condition_dir, folder_name)
                if catalog_path:
                    cond.catalog = load_pose_catalog(catalog_path)
                    cond.catalog_path = catalog_path
                else:
                    print(f"  [warn] no pose-catalog CSV found for {folder_name}; "
                          f"falling back to single refQuat per pose (symmetric variants "
                          f"will under-match)")
                    cond.catalog = {si: pe.quat_wxyz.reshape(1, 4) for si, pe in poses.items()}

                conditions.append(cond)

    return conditions


# ===========================================================================
# Physics world
# ===========================================================================

def find_part_mesh(parts_folder: str, part_name: str) -> str:
    for ext in (".stl", ".obj", ".STL", ".OBJ"):
        candidate = os.path.join(parts_folder, part_name + ext)
        if os.path.isfile(candidate):
            return candidate
    matches = glob.glob(os.path.join(parts_folder, part_name + ".*"))
    if matches:
        return matches[0]
    raise FileNotFoundError(f"Could not find a mesh file for part '{part_name}' in {parts_folder}")


def get_convex_decomposition(part_path: str, cache_dir: str) -> str:
    os.makedirs(cache_dir, exist_ok=True)
    base = os.path.splitext(os.path.basename(part_path))[0]
    out_path = os.path.join(cache_dir, base + "_vhacd.obj")
    log_path = os.path.join(cache_dir, base + "_vhacd_log.txt")
    if not os.path.isfile(out_path):
        print(f"  Running VHACD convex decomposition for {base} ...")
        p.vhacd(part_path, out_path, log_path)
    return out_path


def reset_condition_world(chute_path: str, chute_scale, chute_concave: bool,
                           roll_deg: float, pitch_deg: float, catch_plane: bool):
    p.resetSimulation()
    p.setAdditionalSearchPath(pybullet_data.getDataPath())
    p.setGravity(0, 0, -9.81)
    p.setTimeStep(1.0 / 240.0)

    if catch_plane:
        p.loadURDF("plane.urdf")

    chute_quat_xyzw = build_chute_quat_xyzw(roll_deg, pitch_deg)

    flags = p.GEOM_FORCE_CONCAVE_TRIMESH if chute_concave else 0
    chute_collision = p.createCollisionShape(
        p.GEOM_MESH, fileName=chute_path, meshScale=chute_scale, flags=flags
    )
    chute_visual = p.createVisualShape(
        p.GEOM_MESH, fileName=chute_path, meshScale=chute_scale
    )
    chute_id = p.createMultiBody(
        baseMass=0,
        baseCollisionShapeIndex=chute_collision,
        baseVisualShapeIndex=chute_visual,
        basePosition=[0, 0, 0],
        baseOrientation=chute_quat_xyzw,
    )
    return chute_id, chute_quat_xyzw


def spawn_part(part_collision_path: str, part_scale, mass: float, drop_pos, orientation):
    col = p.createCollisionShape(p.GEOM_MESH, fileName=part_collision_path, meshScale=part_scale)
    vis = p.createVisualShape(p.GEOM_MESH, fileName=part_collision_path, meshScale=part_scale)
    body_id = p.createMultiBody(
        baseMass=mass,
        baseCollisionShapeIndex=col,
        baseVisualShapeIndex=vis,
        basePosition=drop_pos,
        baseOrientation=orientation,
    )
    return body_id


def run_until_settled(body_id, max_steps=2400, lin_thresh=0.005, ang_thresh=0.02,
                       stable_steps_required=60, gui=False, sleep=0.0):
    stable_count = 0
    for step in range(max_steps):
        p.stepSimulation()
        if gui and sleep > 0:
            time.sleep(sleep)
        lin_vel, ang_vel = p.getBaseVelocity(body_id)
        lin_speed = np.linalg.norm(lin_vel)
        ang_speed = np.linalg.norm(ang_vel)
        if lin_speed < lin_thresh and ang_speed < ang_thresh:
            stable_count += 1
            if stable_count >= stable_steps_required:
                return step, True
        else:
            stable_count = 0
    return max_steps, False


# ===========================================================================
# PTFE contact-dynamics sampling
# ===========================================================================

@dataclass
class FrictionModel:
    chute_friction_min: float = 0.05
    chute_friction_max: float = 0.30
    part_friction_min: float = 0.03
    part_friction_max: float = 0.12
    chute_restitution_min: float = 0.03
    chute_restitution_max: float = 0.15
    part_restitution_min: float = 0.03
    part_restitution_max: float = 0.10
    snag_prob: float = 0.05
    snag_friction_boost: float = 4.0
    rolling_friction: float = 0.005
    spinning_friction: float = 0.005

    def sample(self, rng: np.random.Generator):
        chute_fric = rng.uniform(self.chute_friction_min, self.chute_friction_max)
        part_fric = rng.uniform(self.part_friction_min, self.part_friction_max)
        chute_rest = rng.uniform(self.chute_restitution_min, self.chute_restitution_max)
        part_rest = rng.uniform(self.part_restitution_min, self.part_restitution_max)
        snagged = rng.random() < self.snag_prob
        if snagged:
            chute_fric = min(chute_fric * self.snag_friction_boost, 0.9)
        return chute_fric, part_fric, chute_rest, part_rest, snagged


# ===========================================================================
# Per-condition Monte Carlo trial loop
# ===========================================================================

def run_condition_trials(cond: ConditionData, chute_id, chute_quat_xyzw: Tuple[float, float, float, float],
                          part_collision_path: str, args, friction_model: FrictionModel,
                          rng: np.random.Generator, out_dir: str):

    trial_rows = []
    counts: Dict[Optional[int], int] = {}
    n_unsettled = 0
    n_snagged = 0

    drop_local_offset = [args.drop_xy[0], args.drop_xy[1], args.drop_height]
    drop_pos_offset = rotate_vec_by_quat_xyzw(drop_local_offset, chute_quat_xyzw)

    for trial in range(args.n):
        chute_fric, part_fric, chute_rest, part_rest, snagged = friction_model.sample(rng)
        if snagged:
            n_snagged += 1
        p.changeDynamics(chute_id, -1, lateralFriction=chute_fric, restitution=chute_rest,
                          rollingFriction=friction_model.rolling_friction,
                          spinningFriction=friction_model.spinning_friction)

        orn0 = random_quaternion_xyzw(rng)
        drop_pos = [drop_pos_offset[0], drop_pos_offset[1], drop_pos_offset[2]]

        body_id = spawn_part(part_collision_path, [args.part_scale] * 3, args.part_mass,
                              drop_pos, orn0)
        p.changeDynamics(body_id, -1, lateralFriction=part_fric, restitution=part_rest,
                          rollingFriction=friction_model.rolling_friction,
                          spinningFriction=friction_model.spinning_friction)

        steps, settled = run_until_settled(body_id, max_steps=args.max_steps,
                                            gui=args.gui, sleep=args.sleep)
        if not settled:
            n_unsettled += 1

        final_pos, final_orn_world = p.getBasePositionAndOrientation(body_id)
        final_orn_local = rotate_to_chute_local(final_orn_world, chute_quat_xyzw)
        matched_id, match_dist = match_pose_in_catalog(final_orn_local, cond.catalog,
                                                         args.quat_match_tol)

        counts[matched_id] = counts.get(matched_id, 0) + 1

        pe = cond.poses.get(matched_id) if matched_id is not None else None
        trial_rows.append({
            "trial": trial,
            "init_qx": orn0[0], "init_qy": orn0[1], "init_qz": orn0[2], "init_qw": orn0[3],
            "final_x": final_pos[0], "final_y": final_pos[1], "final_z": final_pos[2],
            "final_qx_local": final_orn_local[0], "final_qy_local": final_orn_local[1],
            "final_qz_local": final_orn_local[2], "final_qw_local": final_orn_local[3],
            "settled": settled, "steps_to_settle": steps,
            "matched_pose_id": matched_id if matched_id is not None else -1,
            "match_dist_rad": match_dist,
            "geom_notes": pe.notes if pe else "",
            "geom_ratio_wall": pe.ratio_wall if pe else "",
            "geom_ratio_floor": pe.ratio_floor if pe else "",
            "chute_friction": chute_fric, "part_friction": part_fric,
            "chute_restitution": chute_rest, "part_restitution": part_rest,
            "snagged": snagged,
        })

        p.removeBody(body_id)

        if trial % 100 == 0:
            print(f"    trial {trial}/{args.n}  settled={settled}  "
                  f"matched_pose={matched_id}  dist={match_dist:.4f}")

    os.makedirs(os.path.join(out_dir, cond.part, cond.folder_name), exist_ok=True)
    trials_csv = os.path.join(out_dir, cond.part, cond.folder_name, f"{cond.folder_name}_trials.csv")
    with open(trials_csv, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(trial_rows[0].keys()))
        writer.writeheader()
        writer.writerows(trial_rows)

    print(f"    -> wrote {len(trial_rows)} trials to {trials_csv}")
    print(f"    -> unsettled: {n_unsettled}/{args.n}   snagged: {n_snagged}/{args.n}")

    return counts, n_unsettled


def write_condition_summary(cond: ConditionData, counts: Dict[Optional[int], int], out_dir: str):
    total = sum(counts.values())
    rows = []
    unstable_hits = []

    all_pose_ids = sorted(set(cond.poses.keys()) | {k for k in counts.keys() if k is not None})
    for pose_id in all_pose_ids:
        pe = cond.poses.get(pose_id)
        sim_count = counts.get(pose_id, 0)
        sim_freq = 100.0 * sim_count / total if total else 0.0
        row = {
            "part": cond.part, "alpha_deg": cond.alpha, "beta_deg": cond.beta,
            "pose_id": pose_id,
            "sim_count": sim_count, "sim_freq_pct": sim_freq,
            "geom_CSA_B_pct": pe.csa_b if pe else "",
            "geom_CSA_A_pct": pe.csa_a if pe else "",
            "geom_CSA_N_pct": pe.csa_n if pe else "",
            "geom_CRSA_B_pct": pe.crsa_b if pe else "",
            "geom_CRSA_A_pct": pe.crsa_a if pe else "",
            "geom_CRSA_N_pct": pe.crsa_n if pe else "",
            "ratio_wall": pe.ratio_wall if pe else "",
            "thresh_wall": cond.thresh_wall,
            "ratio_floor": pe.ratio_floor if pe else "",
            "thresh_floor": cond.thresh_floor,
            "geom_notes": pe.notes if pe else "",
            "flagged_unstable_by_geometry": bool(pe and (pe.flag_transitions or pe.flag_floor_unstable)),
        }
        rows.append(row)
        if row["flagged_unstable_by_geometry"] and sim_count > 0:
            unstable_hits.append((pose_id, sim_count))

    # unmatched trials
    unmatched = counts.get(None, 0)
    if unmatched:
        rows.append({
            "part": cond.part, "alpha_deg": cond.alpha, "beta_deg": cond.beta,
            "pose_id": -1, "sim_count": unmatched,
            "sim_freq_pct": 100.0 * unmatched / total if total else 0.0,
            "geom_CSA_B_pct": "", "geom_CSA_A_pct": "", "geom_CSA_N_pct": "",
            "geom_CRSA_B_pct": "", "geom_CRSA_A_pct": "", "geom_CRSA_N_pct": "",
            "ratio_wall": "", "thresh_wall": cond.thresh_wall,
            "ratio_floor": "", "thresh_floor": cond.thresh_floor,
            "geom_notes": "UNMATCHED (no catalog pose within quat_match_tol)",
            "flagged_unstable_by_geometry": False,
        })

    summary_csv = os.path.join(out_dir, cond.part, cond.folder_name,
                                f"{cond.folder_name}_sim_summary.csv")
    with open(summary_csv, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)

    print(f"    -> wrote condition summary to {summary_csv}")
    if unstable_hits:
        print(f"    [!] simulation matched poses the geometric analysis flagged as "
              f"unstable/transitioning: {unstable_hits}")

    return rows


# ===========================================================================
# Main
# ===========================================================================

def main():
    ap = argparse.ArgumentParser(description=__doc__,
                                  formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--parts-folder", required=True,
                     help="Folder containing part STL/OBJ files and a 'results/' subfolder "
                          "produced by ChutePoseAnalysis.m")
    ap.add_argument("--chute", required=True, help="Path to chute mesh (.obj/.stl)")
    ap.add_argument("--chute-scale", type=float, default=1.0)
    ap.add_argument("--chute-concave", action="store_true")
    ap.add_argument("--no-catch-plane", action="store_true",
                     help="Disable the flat plane.urdf floor beneath the chute")

    ap.add_argument("--parts", default=None,
                     help="Comma-separated subset of part names to run (default: all found)")
    ap.add_argument("--alphas", default="15,17.5,20",
                     help="Comma-separated roll angles (deg)")
    ap.add_argument("--betas", default="15,25,35,45",
                     help="Comma-separated pitch angles (deg)")

    ap.add_argument("--n", type=int, default=1000, help="Drop trials per (part, angle) condition")
    ap.add_argument("--seed", type=int, default=None)
    ap.add_argument("--out-dir", default="batch_drop_results")

    ap.add_argument("--part-mass", type=float, default=0.01)
    ap.add_argument("--part-scale", type=float, default=1.0)
    ap.add_argument("--drop-height", type=float, default=0.3,
                     help="Drop height (m) above chute origin, along the chute's local +Z")
    ap.add_argument("--drop-xy", type=float, nargs=2, default=[0.0, 0.0],
                     help="Drop XY offset (m) in the chute's local frame")
    ap.add_argument("--quat-match-tol", type=float, default=0.05,
                     help="Geodesic distance (rad) tolerance for pose matching; "
                          "should match ChutePoseAnalysis.m's quatMatchTol (default 0.05)")
    ap.add_argument("--max-steps", type=int, default=2400)

    ap.add_argument("--gui", action="store_true")
    ap.add_argument("--sleep", type=float, default=0.0)
    ap.add_argument("--dry-run", action="store_true",
                     help="Only discover and print conditions; do not simulate")

    # PTFE contact-dynamics bands
    ap.add_argument("--chute-friction-min", type=float, default=0.05)
    ap.add_argument("--chute-friction-max", type=float, default=0.30)
    ap.add_argument("--part-friction-min", type=float, default=0.03)
    ap.add_argument("--part-friction-max", type=float, default=0.12)
    ap.add_argument("--chute-restitution-min", type=float, default=0.03)
    ap.add_argument("--chute-restitution-max", type=float, default=0.15)
    ap.add_argument("--part-restitution-min", type=float, default=0.03)
    ap.add_argument("--part-restitution-max", type=float, default=0.10)
    ap.add_argument("--snag-prob", type=float, default=0.05)
    ap.add_argument("--snag-friction-boost", type=float, default=4.0)
    ap.add_argument("--rolling-friction", type=float, default=0.005)
    ap.add_argument("--spinning-friction", type=float, default=0.005)

    args = ap.parse_args()

    alphas = [float(a) for a in args.alphas.split(",")]
    betas = [float(b) for b in args.betas.split(",")]
    parts_filter = [s.strip() for s in args.parts.split(",")] if args.parts else None

    print("Discovering conditions ...")
    conditions = discover_conditions(args.parts_folder, alphas, betas, parts_filter)
    print(f"Found {len(conditions)} runnable (part, alpha, beta) condition(s):")
    for c in conditions:
        print(f"  {c.part}: alpha={c.alpha} beta={c.beta} -> {c.folder_name} "
              f"({len(c.poses)} stable poses"
              f"{', catalog=' + os.path.basename(c.catalog_path) if c.catalog_path else ', NO CATALOG'})")

    if args.dry_run or not conditions:
        return

    friction_model = FrictionModel(
        chute_friction_min=args.chute_friction_min, chute_friction_max=args.chute_friction_max,
        part_friction_min=args.part_friction_min, part_friction_max=args.part_friction_max,
        chute_restitution_min=args.chute_restitution_min, chute_restitution_max=args.chute_restitution_max,
        part_restitution_min=args.part_restitution_min, part_restitution_max=args.part_restitution_max,
        snag_prob=args.snag_prob, snag_friction_boost=args.snag_friction_boost,
        rolling_friction=args.rolling_friction, spinning_friction=args.spinning_friction,
    )

    rng = np.random.default_rng(args.seed)
    mode = p.GUI if args.gui else p.DIRECT
    p.connect(mode)

    os.makedirs(args.out_dir, exist_ok=True)
    vhacd_cache_dir = os.path.join(args.out_dir, "_vhacd_cache")

    combined_rows = []
    part_convex_cache: Dict[str, str] = {}

    for cond in conditions:
        print(f"\n=== {cond.part}  alpha={cond.alpha}  beta={cond.beta}  "
              f"({cond.folder_name}) ===")

        if cond.part not in part_convex_cache:
            part_mesh_path = find_part_mesh(args.parts_folder, cond.part)
            part_convex_cache[cond.part] = get_convex_decomposition(part_mesh_path, vhacd_cache_dir)
        part_collision_path = part_convex_cache[cond.part]

        chute_id, chute_quat_xyzw = reset_condition_world(
            args.chute, [args.chute_scale] * 3, args.chute_concave,
            cond.alpha, cond.beta, catch_plane=not args.no_catch_plane,
        )

        counts, n_unsettled = run_condition_trials(
            cond, chute_id, chute_quat_xyzw, part_collision_path,
            args, friction_model, rng, args.out_dir,
        )

        combined_rows.extend(write_condition_summary(cond, counts, args.out_dir))

    p.disconnect()

    combined_csv = os.path.join(args.out_dir, "combined_summary.csv")
    if combined_rows:
        with open(combined_csv, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=list(combined_rows[0].keys()))
            writer.writeheader()
            writer.writerows(combined_rows)
        print(f"\nDone. Combined summary across all conditions: {combined_csv}")


if __name__ == "__main__":
    main()